# B6 — Agent d'investigation : une alerte entre, un rapport d'audit sort

**Ce que fait l'agent** (`audit/agent.py`, `audit/agent_tools.py`)
1. **Lit** l'alerte (contrat d'interface de A).
2. **Appelle 4 outils** : voisinage (transactions voisines réelles d'A), explication (Integrated Gradients de B5), statistiques du voisinage, recherche réglementaire (RAG).
3. **Rédige** le rapport avec le LLM-LoRA de B4 (rang 16).
4. **Contrôle** chaque déduction du rapport (niveau de risque, fiabilité, note de prudence, action du schéma, recommandations GAFI) et **remplace** les sections incohérentes par la version calculée ; chaque correction est tracée.
5. **Annexe** les éléments recueillis par les outils (section 6) et la traçabilité.

**Pourquoi le contrôle (étape 4)** : B4 a montré que le LoRA recopie bien les faits (100 %) mais déduit mal (14 % de rapports cohérents à r=16).

**Comparaison avec / sans RAG** (cahier B6), sur les mêmes alertes :
| Mode | Le RAG est… |
|---|---|
| `off` | absent |
| `prompt` | ajouté au prompt du LLM (comme B4) |
| `annex` | copié tel quel dans la section 6, hors du LLM |

**Données** : alertes **réelles** d'A (modèle FedAvg, confiance calibrée par B5). Rapports de référence = gabarits calculés (cibles synthétiques) à partir de ces alertes.

**Limites à écrire dans l'analyse** : planification fixe (ce n'est pas le LLM qui choisit ses outils) ; règles de contrôle = conventions du jeu de B3 (seuils de risque sur la confiance calibrée, pas des règles réglementaires validées) ; les statistiques sont calculées sur le voisinage fourni par A, pas sur l'historique complet des comptes. Le « 100 % cohérent » du rapport final est vrai **par construction** : l'indicateur utile est la part de sections que le LLM a écrites correctement lui-même.

**Avant de lancer** : GPU T4 ; fichiers `audit/` poussés sur la branche ; `audit/results/b5/alerts_explained.jsonl` et `audit/instruction_data/` (v2) poussés ; adaptateur `audit/lora_adapters/rank_16` **réentraîné en B4 sur le jeu v2** disponible, sinon la cellule 3 te demande `b4_outputs.zip`.

In [ ]:
# ===== PARAMÈTRES =====
REPO_URL   = "https://github.com/hafid700/FinTrust-AI-Project.git"
BRANCH     = "feature-B"
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"   # le même que B4
ADAPTER_RANK = 16                           # adaptateur de B4 à utiliser
N_CASES    = 20      # cahier : 10 à 20 cas (répartis sur le jeu de test pour couvrir les 4 schémas)
A_BRANCH   = "feature-A"   # contient reports/a6/neighbor_transactions.csv
RAG_K      = 2       # extraits réglementaires par alerte
EVAL_BS    = 8
SEED       = 42

## 1. Installation, GPU, dépôt

In [ ]:
!pip -q install -U transformers peft accelerate sentence-transformers faiss-cpu pypdf pandas

In [ ]:
import importlib.metadata as md_
import torch
for pkg in ["torch", "transformers", "peft", "accelerate", "sentence-transformers", "faiss-cpu", "pypdf"]:
    print(f"{pkg}=={md_.version(pkg)}")
assert torch.cuda.is_available(), "Pas de GPU : Exécution > Modifier le type d'exécution > T4 GPU"
print("GPU :", torch.cuda.get_device_name(0))

In [ ]:
import os
if not os.path.isdir("FinTrust-AI-Project"):
    !git clone -b {BRANCH} {REPO_URL}
%cd FinTrust-AI-Project
!ls audit
# Voisinage réel (fichier d'A)
if not os.path.isdir("../A_repo"):
    !git clone -q --depth 1 -b {A_BRANCH} {REPO_URL} ../A_repo
NEIGHBOR_CSV = "../A_repo/reports/a6/neighbor_transactions.csv"
assert os.path.isfile(NEIGHBOR_CSV), "neighbor_transactions.csv introuvable sur la branche d'A"

## 2. Adaptateur LoRA de B4
L'agent réutilise l'adaptateur déjà entraîné. S'il n'est pas dans le dépôt, importe `b4_outputs.zip` (téléchargé à la fin de B4) : il sera décompressé dans `audit/`.

In [ ]:
ADAPTER_DIR = f"audit/lora_adapters/rank_{ADAPTER_RANK}"
if not os.path.isdir(ADAPTER_DIR):
    from google.colab import files
    print("Adaptateur absent : importe b4_outputs.zip")
    up = files.upload()
    zip_name = next(iter(up))
    !unzip -qo "{zip_name}" -d audit
assert os.path.isfile(f"{ADAPTER_DIR}/adapter_config.json"), "adaptateur introuvable"
print("adaptateur OK :", ADAPTER_DIR)

## 3. Corpus réglementaire et index FAISS (B1)
Même procédure que B4, avec le même contrôle : si le GAFI ou les textes marocains manquent, l'outil de recherche ramènerait des passages hors sujet.

In [ ]:
import requests
os.makedirs("audit/corpus", exist_ok=True)
CORPUS = {
    "FATF_Recommendations_2012.pdf": "https://www.fatf-gafi.org/content/dam/fatf-gafi/recommendations/FATF%20Recommendations%202012.pdf.coredownload.inline.pdf",
    "RGPD.pdf": "https://wallex.wallonie.be/files/medias/10/RGPD.pdf",
    "Guide-LBC-FT-FR-03_25.pdf": "https://cnasnu.justice.gov.ma/wp-content/uploads/2025/03/Guide-LBC-FT-FR-03_25.pdf",
    "Lutte-contre-le-blanchiment-de-capitaux.pdf": "https://adala.justice.gov.ma/api/uploads/2024/02/29/Lutte%20contre%20le%20blanchiment%20de%20capitaux-1709213287982.pdf",
}
for name, url in CORPUS.items():
    path = f"audit/corpus/{name}"
    if os.path.exists(path) and os.path.getsize(path) > 10_000:
        print("déjà présent :", name)
        continue
    try:
        r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=120)
        r.raise_for_status()
        open(path, "wb").write(r.content)
        print("OK     :", name, len(r.content) // 1024, "Ko")
    except Exception as e:
        print("ÉCHEC  :", name, "->", e, "| dépose le PDF à la main dans audit/corpus/")
if not os.path.exists("audit/rag_index/index.faiss"):
    !python audit/build_rag_index.py
else:
    print("index FAISS déjà construit")

In [ ]:
import collections, json
with open("audit/rag_index/chunks.jsonl", encoding="utf-8") as f:
    by_source = collections.Counter(json.loads(line)["source"] for line in f)
for name, n in by_source.items():
    print(f"{n:4d} chunks  {name}")
has_fatf  = any("fatf" in s.lower() for s in by_source)
has_maroc = any(("lutte" in s.lower() or "guide" in s.lower()) for s in by_source)
assert has_fatf and has_maroc, "Il manque le GAFI ou les textes marocains dans l'index : relance la cellule précédente."

## 4. Modèle, données et outils
Le backend est **réel** (`RealBackend`) : voisins lus dans `neighbor_transactions.csv` (A), explications lues dans `alerts_explained.jsonl` (B5). Aucune valeur simulée.

In [ ]:
import sys, re, time, gc
sys.path.insert(0, "audit")
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
import lora_utils as L
import agent as A
import agent_tools as T
from search_rag import search

L.set_seed(SEED)
os.makedirs("audit/results", exist_ok=True)
train_recs = L.load_jsonl("audit/instruction_data/train.jsonl")
all_test   = L.load_jsonl("audit/instruction_data/test.jsonl")
step       = max(1, len(all_test) // N_CASES)
test_recs  = all_test[::step][:N_CASES]   # répartis, pas les N premiers
alerts = [json.loads(r["input"]) for r in test_recs]
by_tx  = {a["tx_id"]: r for a, r in zip(alerts, test_recs)}

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
stats = L.token_stats(L.build_dataset(tok, train_recs))
MAX_NEW_TOKENS = int(stats["report_tokens_max"] * 1.2)
print("répartition des schémas :", collections.Counter(a["pattern"] for a in alerts))
print("cas :", len(alerts), "| max_new_tokens =", MAX_NEW_TOKENS)

try:
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.float16)
except TypeError:
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16)
model = L.load_lora(base.to("cuda"), ADAPTER_DIR)

explained = [json.loads(l) for l in open("audit/results/b5/alerts_explained.jsonl", encoding="utf-8")]
backend = T.RealBackend(NEIGHBOR_CSV, explained)
tools = T.make_tools(backend, search, k=RAG_K)
print("outils :", list(tools))

## 5. Un cas, de bout en bout
On regarde la **trace des outils** et le **rapport final** pour une alerte. Les sections marquées dans la traçabilité ont été remplacées par les règles.

In [ ]:
def generate_fn(batch_alerts, contexts):
    """LLM-LoRA : un brouillon (texte, terminé) par alerte, par lots, greedy."""
    prompts = []
    for a in batch_alerts:
        r = by_tx[a["tx_id"]]
        ctx = contexts[a["tx_id"]] if contexts else None
        prompts.append(L.prompt_text(tok, r["input"], r["instruction"], ctx))
    texts, fin = L.generate_reports(model, tok, prompts, max_new_tokens=MAX_NEW_TOKENS,
                                    batch_size=EVAL_BS, return_finished=True)
    return list(zip(texts, fin))

demo = A.run_agent(alerts[:1], tools, generate_fn, "annex")[0]
print(json.dumps(demo["trace"], ensure_ascii=False, indent=1))
print("\nsections remplacées :", demo["replaced"], "| remplacement complet :", demo["full_replacement"])
print("\n" + demo["report"])

## 6. Comparaison avec / sans RAG sur les cas choisis
Pour chaque mode : `draft_*` = le brouillon du LLM **avant** contrôle, `final_strict_%` = rapport final (cohérent par construction), `sections_kept_from_llm_%` = part des sections écrites correctement par le LLM, `replaced_*_%` = quelles sections il rate. Les statistiques `avg_sources` comptent les extraits réglementaires cités dans la section 6.

In [ ]:
CONTEXTS = L.build_contexts(alerts, search, k=RAG_K)
ALL, ROWS = {}, []
for mode in A.RAG_MODES:
    t0 = time.time()
    res = A.run_agent(alerts, tools, generate_fn, mode,
                      contexts=CONTEXTS if mode == "prompt" else None,
                      simulated=backend.simulated)
    ALL[mode] = res
    row = {"mode_rag": mode, **A.summarize_agent(res), "temps_s": round(time.time() - t0)}
    ROWS.append(row)
    print(mode, "->", {k: row[k] for k in ("draft_usable_%", "draft_strict_%", "sections_kept_from_llm_%", "avg_sources")})
    pd.DataFrame(ROWS).to_csv("audit/results/b6_comparison.csv", index=False)  # sauvegarde incrémentale
df = pd.DataFrame(ROWS)
df

## 7. Sauvegarde et échantillon pour la relecture manuelle
Pour chaque mode : rapports complets (`b6_reports_<mode>.jsonl`, avec brouillon, rapport final, trace et mesures) et un échantillon Markdown à noter à la main (clarté, citations justes).

In [ ]:
for mode, res in ALL.items():
    with open(f"audit/results/b6_reports_{mode}.jsonl", "w", encoding="utf-8") as f:
        for r in res:
            f.write(json.dumps({k: r[k] for k in ("id", "draft", "report", "replaced", "full_replacement", "trace", "n_sources")}
                               | {"before": {k: v for k, v in r["before"].items() if k != "problems"},
                                  "after": {k: v for k, v in r["after"].items() if k != "problems"}},
                               ensure_ascii=False, default=str) + "\n")

lines = ["# Échantillon B6 pour relecture manuelle", ""]
for mode in A.RAG_MODES:
    lines += [f"## Mode RAG : {mode}", ""]
    for r in ALL[mode][:3]:
        lines += [f"### {r['id']}", f"Sections remplacées : {[s for s in r['replaced']]} | complet : {r['full_replacement']}",
                  "", "```", r["report"], "```", "", "Grille : clarté __/5 | citations justes __/5 | remarques :", ""]
open("audit/results/b6_samples.md", "w", encoding="utf-8").write("\n".join(lines))
print("écrit : b6_comparison.csv, b6_reports_*.jsonl, b6_samples.md")

In [ ]:
from google.colab import files
!cd audit && zip -qr b6_outputs.zip results/b6_*
files.download("audit/b6_outputs.zip")

## 8. Limites à reprendre dans l'analyse
- **Planification fixe** : les outils sont appelés dans un ordre prédéfini ; ce n'est pas un agent qui raisonne sur ses outils.
- **Contrôle = règles de B3** : niveau de risque (confiance calibrée : 0,90 / 0,70 / 0,50), fiabilité du schéma et recommandations sont des conventions, pas des règles réglementaires validées par un expert.
- **100 % cohérent par construction** : ne pas présenter `final_strict_%` comme une performance du LLM ; la mesure du LLM est `draft_strict_%` et `sections_kept_from_llm_%`.
- **Statistiques du voisinage** : calculées sur les voisines fournies par A, pas sur l'historique complet du compte ; l'explication ne couvre que 45 % du poids du modèle (55 % vient du contexte réseau, voir B5).
- **RAG** : en mode `annex` les extraits sont copiés de l'index (pas d'invention) mais ne sont pas « compris » par le LLM ; un RAG qui améliore la rédaction suppose un entraînement avec contexte (voir B4).
- **20 cas, une graine** : écarts de quelques rapports non significatifs.